---
## 📦 Step 0 — Install dependencies & upload files

In [ ]:
# ── Install required libraries ──────────────────────────────────────────────
!pip install openpyxl xlsxwriter -q

import pandas as pd
import numpy as nps
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Pretty display
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print('✅  Libraries loaded')

In [ ]:
# ── Upload files (run this cell then click Choose Files) ────────────────────
from google.colab import files
print('Upload: Classification.xlsx, Consumption.xlsx, Lead_Time.xlsx')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

---
## ⚙️ Step 1 — Configuration (edit parameters here)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║          CONFIGURATION — ADJUST BEFORE RUNNING              ║
# ╚══════════════════════════════════════════════════════════════╝

# --- File names -----------------------------------------------------------
FILE_CLASSIFICATION = 'Classification.xlsx'
FILE_CONSUMPTION    = 'Consumption.xlsx'
FILE_LEAD_TIME      = 'Lead_Time.xlsx'

# --- Cost parameters (TND) ------------------------------------------------
ORDER_COST     = 150    # Fixed cost per purchase order (TND)
HOLDING_RATE   = 0.20   # Annual holding cost as % of unit value (20%)

# --- Service levels by ABC class ------------------------------------------
# Higher service level = more safety stock but fewer stockouts
SERVICE_LEVELS = {
    'B': {'z': 1.96, 'label': '97.5%'},   # B class → high criticality
    'C': {'z': 1.65, 'label': '95.0%'},   # C class → standard
}

# --- HML definitions (active consumption months in 2025) ------------------
# H = 6+ months, M = 3-5 months, L = 1-2 months
# (Already classified in your file — used here for strategy)
HML_STRATEGY = {
    'H': 'Continuous Review  — tight ROP control',
    'M': 'Periodic Review    — monthly check',
    'L': 'Min-Max            — simple reorder policy',
}

# --- Calendar -------------------------------------------------------------
WORKING_DAYS_YEAR  = 250   # Working days per year
WORKING_DAYS_MONTH = 21    # Approx working days per month

# --- French month name parser ---------------------------------------------
FRENCH_MONTHS = {
    'janv':1,'févr':2,'mars':3,'avr':4,'mai':5,'juin':6,
    'juil':7,'août':8,'sept':9,'oct':10,'nov':11,'déc':12
}

print('✅  Configuration set')
print(f'   Order cost   : TND {ORDER_COST}')
print(f'   Holding rate : {HOLDING_RATE*100:.0f}%/year')
for cls, sl in SERVICE_LEVELS.items():
    print(f'   Service level {cls}: {sl["label"]} (Z={sl["z"]})')

---
## 📂 Step 2 — Load & validate raw data

In [ ]:
# ── 2.1 Load classification ──────────────────────────────────────────────
df_cls = pd.read_excel(FILE_CLASSIFICATION)
df_cls.columns = df_cls.columns.str.strip()
df_cls = df_cls.rename(columns={'Unit.': 'Unit', 'PMP': 'UnitCost'})
df_cls['Article'] = df_cls['Article'].astype(str).str.strip()

print('=== CLASSIFICATION ===')
print(f'Shape : {df_cls.shape}')
print(f'Columns: {df_cls.columns.tolist()}')
display(df_cls.head(5))

print('\nABC distribution:')
display(df_cls['ABC'].value_counts().rename('Count').to_frame())
print('\nHML distribution:')
display(df_cls['HML'].value_counts().rename('Count').to_frame())

In [ ]:
# ── 2.2 Load consumption ─────────────────────────────────────────────────
df_cons_raw = pd.read_excel(FILE_CONSUMPTION, sheet_name='Consumption')
df_cons_raw.columns = df_cons_raw.columns.str.strip()
df_cons_raw['Article'] = df_cons_raw['Article'].astype(str).str.strip()

print('=== CONSUMPTION ===')
print(f'Shape: {df_cons_raw.shape}')
print(f'Columns: {df_cons_raw.columns.tolist()}')
display(df_cons_raw.head(5))

# Parse French month strings → proper dates
def parse_french_month(s):
    try:
        parts = str(s).strip().split('-')
        m = FRENCH_MONTHS.get(parts[0].lower())
        y = int('20' + parts[1]) if len(parts[1]) == 2 else int(parts[1])
        if m:
            return pd.Timestamp(year=y, month=m, day=1)
    except Exception:
        pass
    return pd.NaT

df_cons_raw['Date'] = df_cons_raw['Month'].apply(parse_french_month)
df_cons_raw['Consumption'] = df_cons_raw['Valeur'].abs()   # negatives = withdrawals
df_cons_raw = df_cons_raw[df_cons_raw['Date'].notna()].copy()

print(f'\nDate range: {df_cons_raw["Date"].min().strftime("%b-%Y")} → {df_cons_raw["Date"].max().strftime("%b-%Y")}')
print(f'Unique articles: {df_cons_raw["Article"].nunique()}')

In [ ]:
# ── 2.3 Load lead times ──────────────────────────────────────────────────
df_lt = pd.read_excel(FILE_LEAD_TIME, sheet_name='Lead Time')
df_lt.columns = df_lt.columns.str.strip()
df_lt = df_lt.rename(columns={'Lead Time (Days)': 'LeadTime', 'Local /Import': 'Source'})
df_lt['Article'] = df_lt['Article'].astype(str).str.strip()

print('=== LEAD TIMES ===')
print(f'Shape: {df_lt.shape}')
display(df_lt.head(5))
print('\nLead time statistics (days):')
display(df_lt['LeadTime'].describe().rename('Days').to_frame())
print('\nSource distribution:')
display(df_lt['Source'].value_counts().rename('Count').to_frame())

---
## 📊 Step 3 — Demand statistics per article

In [ ]:
# ── 3.1 Monthly pivot table: articles × months ───────────────────────────
pivot = (
    df_cons_raw
    .pivot_table(index='Article', columns='Date', values='Consumption', aggfunc='sum')
    .fillna(0)
)

print(f'Pivot shape: {pivot.shape}  ({pivot.shape[0]} articles × {pivot.shape[1]} months)')
display(pivot.head(3))

In [ ]:
# ── 3.2 Demand stats ─────────────────────────────────────────────────────
stats = pd.DataFrame(index=pivot.index)
stats['TotalConsumption']  = pivot.sum(axis=1)
stats['ActiveMonths']      = (pivot > 0).sum(axis=1)
stats['AvgMonthlyDemand']  = pivot.mean(axis=1)
stats['StdMonthlyDemand']  = pivot.std(axis=1, ddof=1).fillna(0)
stats['MaxMonthlyDemand']  = pivot.max(axis=1)
stats['MinNonZeroDemand']  = pivot.replace(0, np.nan).min(axis=1)

# Convert to daily
stats['AvgDailyDemand']    = stats['AvgMonthlyDemand'] / WORKING_DAYS_MONTH
stats['StdDailyDemand']    = stats['StdMonthlyDemand'] / np.sqrt(WORKING_DAYS_MONTH)

# Coefficient of Variation → XYZ
stats['CV'] = np.where(
    stats['AvgMonthlyDemand'] > 0,
    stats['StdMonthlyDemand'] / stats['AvgMonthlyDemand'],
    np.nan
)

def xyz_class(cv):
    if pd.isna(cv):  return 'Z'
    if cv <= 0.25:   return 'X'   # Stable
    if cv <= 0.50:   return 'Y'   # Variable
    return 'Z'                     # Erratic / intermittent

stats['XYZ'] = stats['CV'].apply(xyz_class)

print('Demand statistics computed for', len(stats), 'articles')
print('\nXYZ distribution:')
print(stats['XYZ'].value_counts())
display(stats.describe())

In [ ]:
# ── 3.3 Visualise demand patterns ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Demand Pattern Analysis — All Articles', fontsize=14, fontweight='bold')

# CV distribution
cv_vals = stats['CV'].dropna()
axes[0].hist(cv_vals.clip(upper=3), bins=40, color='#2E75B6', edgecolor='white')
axes[0].axvline(0.25, color='green',  linestyle='--', label='X/Y boundary (0.25)')
axes[0].axvline(0.50, color='orange', linestyle='--', label='Y/Z boundary (0.50)')
axes[0].set_title('Coefficient of Variation Distribution')
axes[0].set_xlabel('CV (capped at 3)')
axes[0].set_ylabel('Number of Articles')
axes[0].legend(fontsize=8)

# XYZ pie
xyz_counts = stats['XYZ'].value_counts()
colors_xyz = ['#70AD47', '#FFD966', '#FF4C4C']
axes[1].pie(xyz_counts, labels=[f'{k}\n({v})' for k,v in xyz_counts.items()],
            autopct='%1.1f%%', colors=colors_xyz, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('XYZ Classification')

# Active months histogram
axes[2].hist(stats['ActiveMonths'], bins=range(0, 38), color='#5B9BD5', edgecolor='white')
axes[2].axvline(6,  color='green',  linestyle='--', label='H threshold (6 mo)')
axes[2].axvline(3,  color='orange', linestyle='--', label='M threshold (3 mo)')
axes[2].set_title('Active Months per Article (36-month window)')
axes[2].set_xlabel('Months with Consumption > 0')
axes[2].set_ylabel('Number of Articles')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('demand_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → demand_patterns.png')

---
## 🔗 Step 4 — Merge master data & ABC × HML analysis

In [ ]:
# ── 4.1 Build master dataframe ───────────────────────────────────────────
master = (
    stats.reset_index()
    .merge(df_cls[['Article','Désignation article','Unit','ABC','HML','UnitCost']],
           on='Article', how='left')
    .merge(df_lt[['Article','LeadTime','Source']], on='Article', how='left')
)

# Fill missing
median_lt = df_lt['LeadTime'].median()
master['ABC']      = master['ABC'].fillna('C')
master['HML']      = master['HML'].fillna('L')
master['UnitCost'] = master['UnitCost'].fillna(0)
master['LeadTime'] = master['LeadTime'].fillna(median_lt)

# Annual demand value for reference
master['AnnualDemandValue'] = master['AvgMonthlyDemand'] * 12 * master['UnitCost']

print(f'Master dataset: {master.shape[0]} articles')
display(master[['Article','Désignation article','Unit','ABC','HML','UnitCost',
                'AvgMonthlyDemand','StdMonthlyDemand','LeadTime','XYZ']].head(10))

In [ ]:
# ── 4.2 ABC × HML matrix ─────────────────────────────────────────────────
matrix_count = pd.crosstab(master['ABC'], master['HML'],
                            margins=True, margins_name='TOTAL')
matrix_value = pd.crosstab(master['ABC'], master['HML'],
                            values=master['AnnualDemandValue'],
                            aggfunc='sum', margins=True, margins_name='TOTAL')

print('=== ABC × HML — Article Counts ===')
display(matrix_count)
print('\n=== ABC × HML — Annual Demand Value (TND) ===')
display(matrix_value.style.format('{:,.0f}'))

In [ ]:
# ── 4.3 ABC × HML heatmap + management strategy ──────────────────────────

# Strategy matrix
strategy = {
    ('B','H'): 'CRITICAL\nTight ROP\nShort review',
    ('B','M'): 'IMPORTANT\nROP + SS\nMid review',
    ('B','L'): 'MONITOR\nMin-Max\nLow SS',
    ('C','H'): 'ACTIVE\nROP-based\nStandard SS',
    ('C','M'): 'STANDARD\nPeriodic\nLight SS',
    ('C','L'): 'BASIC\nMin-Max\nMin SS',
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('ABC × HML Analysis', fontsize=14, fontweight='bold')

# Heatmap — article counts
cnt = matrix_count.drop('TOTAL').drop('TOTAL', axis=1).reindex(['B','C'], axis=0).reindex(['H','M','L'], axis=1)
sns.heatmap(cnt.astype(float), annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=1, ax=axes[0], cbar_kws={'label': 'Article Count'},
            annot_kws={'size': 14, 'weight': 'bold'})
axes[0].set_title('Article Counts per Cell')
axes[0].set_xlabel('HML (Movement Frequency)')
axes[0].set_ylabel('ABC (Value Class)')

# Strategy grid
colors_map = {
    ('B','H'):'#C00000', ('B','M'):'#FF4C4C', ('B','L'):'#FF9999',
    ('C','H'):'#FF9900', ('C','M'):'#FFD966', ('C','L'):'#E2EFDA',
}
axes[1].set_xlim(0, 3); axes[1].set_ylim(0, 2)
axes[1].set_xticks([0.5,1.5,2.5]); axes[1].set_xticklabels(['H','M','L'], fontsize=12)
axes[1].set_yticks([0.5,1.5]);     axes[1].set_yticklabels(['C','B'], fontsize=12)
axes[1].set_xlabel('HML (Movement Frequency)', fontsize=11)
axes[1].set_ylabel('ABC (Value Class)', fontsize=11)
axes[1].set_title('Management Strategy per Cell')

for (abc, hml), text in strategy.items():
    row = {'B':1,'C':0}[abc]
    col = {'H':0,'M':1,'L':2}[hml]
    rect = mpatches.FancyBboxPatch((col+0.05, row+0.05), 0.9, 0.9,
                                    boxstyle='round,pad=0.05',
                                    facecolor=colors_map[(abc,hml)],
                                    edgecolor='white', linewidth=2)
    axes[1].add_patch(rect)
    n = matrix_count.loc[abc, hml] if abc in matrix_count.index and hml in matrix_count.columns else 0
    axes[1].text(col+0.5, row+0.65, text, ha='center', va='center',
                 fontsize=7.5, fontweight='bold', color='white' if row==1 and col==0 else '#333333')
    axes[1].text(col+0.5, row+0.15, f'n={n}', ha='center', va='center',
                 fontsize=9, color='#333333', fontstyle='italic')

plt.tight_layout()
plt.savefig('abc_hml_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved → abc_hml_matrix.png')

---
## ⚙️ Step 5 — Safety Stock, ROP, EOQ, Min, Max calculation

| Formula | Description |
|---|---|
| `SS = Z × σd × √LT` | Safety Stock: Z-factor × std daily demand × √lead time |
| `ROP = D̄ × LT + SS` | Reorder Point: avg daily demand × lead time + safety stock |
| `EOQ = √(2 × D_annual × S / H)` | Economic Order Quantity |
| `Stock Min = ROP` | Trigger a reorder when stock hits this level |
| `Stock Max = ROP + EOQ` | Order-up-to level |

In [ ]:
# ── 5.1 Compute planning parameters per article ──────────────────────────

def compute_params(row):
    abc       = row['ABC']
    z         = SERVICE_LEVELS.get(abc, SERVICE_LEVELS['C'])['z']
    sl_label  = SERVICE_LEVELS.get(abc, SERVICE_LEVELS['C'])['label']
    d_avg     = row['AvgDailyDemand']    # units/day
    d_std     = row['StdDailyDemand']    # std dev daily demand
    lt        = max(row['LeadTime'], 1)  # days (min 1 to avoid zero)
    cost      = row['UnitCost']

    # Safety Stock: Z × σd × √LT
    ss = z * d_std * np.sqrt(lt)

    # Reorder Point
    rop = (d_avg * lt) + ss

    # EOQ — Economic Order Quantity
    d_annual     = d_avg * WORKING_DAYS_YEAR
    holding_unit = cost * HOLDING_RATE if cost > 0 else 1.0
    if d_annual > 0:
        eoq = np.sqrt(2 * d_annual * ORDER_COST / holding_unit)
    else:
        eoq = 0.0

    stock_min = rop
    stock_max = rop + eoq

    # Orders per year
    orders_yr = d_annual / eoq if eoq > 0 else 0

    # Average stock investment (cycle stock + safety stock)
    avg_stock_value = ((eoq / 2) + ss) * cost

    return pd.Series({
        'Z_Factor':         round(z, 2),
        'ServiceLevel':     sl_label,
        'SafetyStock':      round(ss, 2),
        'ROP':              round(rop, 2),
        'EOQ':              round(eoq, 2),
        'StockMin':         round(stock_min, 2),
        'StockMax':         round(stock_max, 2),
        'OrdersPerYear':    round(orders_yr, 1),
        'AvgStockValue_TND':round(avg_stock_value, 2),
    })

params = master.apply(compute_params, axis=1)
master = pd.concat([master, params], axis=1)

print('✅  Parameters computed')
print(f"   Avg Safety Stock : {master['SafetyStock'].mean():,.1f} units")
print(f"   Avg ROP          : {master['ROP'].mean():,.1f} units")
print(f"   Avg EOQ          : {master['EOQ'].mean():,.1f} units")
print(f"   Total SS Value   : TND {(master['SafetyStock'] * master['UnitCost']).sum():,.0f}")

display(master[['Article','Désignation article','ABC','HML','XYZ',
                'ServiceLevel','SafetyStock','ROP','EOQ',
                'StockMin','StockMax','AvgStockValue_TND']].head(10))

In [ ]:
# ── 5.2 Parameters summary by ABC × HML ─────────────────────────────────
summary_params = (
    master.groupby(['ABC','HML'])
    .agg(
        Articles        = ('Article',          'count'),
        Avg_SS          = ('SafetyStock',       'mean'),
        Avg_ROP         = ('ROP',               'mean'),
        Avg_EOQ         = ('EOQ',               'mean'),
        Total_SS_Value  = ('AvgStockValue_TND', 'sum'),
        Avg_LeadTime    = ('LeadTime',          'mean'),
    )
    .round(1)
    .reset_index()
)

print('=== Planning Parameters by ABC × HML ===')
display(summary_params)

In [ ]:
# ── 5.3 Visualise SS, ROP, EOQ by segment ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Planning Parameters by ABC × HML Segment', fontsize=13, fontweight='bold')

metrics   = ['Avg_SS', 'Avg_ROP', 'Avg_EOQ']
titles    = ['Average Safety Stock (units)', 'Average ROP (units)', 'Average EOQ (units)']
palette   = {'B': '#C00000', 'C': '#2E75B6'}

for ax, metric, title in zip(axes, metrics, titles):
    for abc in ['B', 'C']:
        subset = summary_params[summary_params['ABC'] == abc].sort_values('HML')
        ax.bar(
            [f"{abc}-{h}" for h in subset['HML']],
            subset[metric],
            color=palette[abc], alpha=0.85,
            label=f'ABC={abc}', edgecolor='white'
        )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('ABC-HML Segment')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend()

plt.tight_layout()
plt.savefig('planning_params.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔄 Step 6 — Stock simulation (36-month replay)

In [ ]:
# ── 6.1 Simulate each article over 36 months ─────────────────────────────
# Logic:
#  - Start stock at ROP + EOQ/2 (middle of cycle)
#  - Each month: deduct demand, check vs ROP
#  - If stock ≤ ROP and EOQ > 0 → place order (receive immediately for sim)
#  - Count stockout months (stock went negative before reorder)

print('Running simulation …')
sim_records = []

for _, row in master.iterrows():
    art = row['Article']
    rop = row['ROP']
    eoq = row['EOQ']

    # Monthly consumption series
    if art in pivot.index:
        art_series = pivot.loc[art]
    else:
        art_series = pd.Series(dtype=float)

    stock     = rop + (eoq / 2) if eoq > 0 else rop * 2
    stockouts = 0
    orders    = 0
    total_dem = 0
    fill_qty  = 0

    for date, demand in art_series.items():
        stock -= demand
        total_dem += demand

        if stock < 0:
            stockouts += 1
            fill_qty  += abs(stock)   # unfilled demand
            stock      = 0

        if stock <= rop and eoq > 0:
            stock  += eoq
            orders += 1

    n_months   = len(art_series)
    stockout_r = round(stockouts / n_months * 100, 1) if n_months > 0 else 0
    fill_rate  = round((1 - fill_qty / total_dem) * 100, 1) if total_dem > 0 else 100.0

    sim_records.append({
        'Article':          art,
        'Stockout_Months':  stockouts,
        'Stockout_Rate_Pct':stockout_r,
        'Fill_Rate_Pct':    fill_rate,
        'Total_Orders':     orders,
        'Avg_Orders_Year':  round(orders / (n_months / 12), 1) if n_months > 0 else 0,
    })

df_sim = pd.DataFrame(sim_records)
master = master.drop(columns=[c for c in master.columns if c in df_sim.columns and c != 'Article'])
master = master.merge(df_sim, on='Article', how='left')

print(f'✅  Simulation complete')
print(f"   Articles with ≥1 stockout : {(master['Stockout_Months']>0).sum()}")
print(f"   Avg fill rate             : {master['Fill_Rate_Pct'].mean():.1f}%")
print(f"   Avg stockout rate         : {master['Stockout_Rate_Pct'].mean():.1f}%")

In [ ]:
# ── 6.2 Simulate & PLOT single article stock trace ───────────────────────
# Change ARTICLE_CODE to any article you want to inspect
ARTICLE_CODE = master.sort_values('Stockout_Rate_Pct', ascending=False).iloc[0]['Article']
print(f'Plotting stock trace for article: {ARTICLE_CODE}')

row  = master[master['Article'] == ARTICLE_CODE].iloc[0]
rop  = row['ROP']
eoq  = row['EOQ']
ss   = row['SafetyStock']
name = row['Désignation article'] if pd.notna(row['Désignation article']) else ARTICLE_CODE

if ARTICLE_CODE in pivot.index:
    art_series = pivot.loc[ARTICLE_CODE]
    stock_trace = []
    stock = rop + (eoq / 2) if eoq > 0 else rop * 2
    reorder_pts = []

    for date, demand in art_series.items():
        stock -= demand
        if stock < 0:
            stock = 0
        if stock <= rop and eoq > 0:
            stock += eoq
            reorder_pts.append(date)
        stock_trace.append({'Date': date, 'Stock': stock, 'Demand': demand})

    df_trace = pd.DataFrame(stock_trace)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                    gridspec_kw={'height_ratios': [3, 1]})
    fig.suptitle(f'Stock Simulation — {name[:60]}\n(Article {ARTICLE_CODE})',
                 fontsize=12, fontweight='bold')

    ax1.fill_between(df_trace['Date'], df_trace['Stock'], alpha=0.15, color='#2E75B6')
    ax1.plot(df_trace['Date'], df_trace['Stock'], color='#2E75B6', linewidth=2, label='Stock Level')
    ax1.axhline(rop, color='#FF4C4C', linestyle='--', linewidth=1.5, label=f'ROP = {rop:.1f}')
    ax1.axhline(ss,  color='#FF9900', linestyle=':',  linewidth=1.5, label=f'Safety Stock = {ss:.1f}')
    ax1.axhline(row['StockMax'], color='#70AD47', linestyle='--', linewidth=1,
                label=f'Stock Max = {row["StockMax"]:.1f}')
    for rp in reorder_pts:
        ax1.axvline(rp, color='purple', alpha=0.3, linewidth=1)
    ax1.set_ylabel('Stock (units)')
    ax1.legend(fontsize=9, loc='upper right')
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

    ax2.bar(df_trace['Date'], df_trace['Demand'], color='#5B9BD5', width=20, alpha=0.8)
    ax2.set_ylabel('Monthly Demand')
    ax2.set_xlabel('Date')
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

    plt.tight_layout()
    plt.savefig(f'stock_trace_{ARTICLE_CODE}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Article not found in consumption data')

In [ ]:
# ── 6.3 Simulation results overview ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Simulation Results Overview', fontsize=13, fontweight='bold')

# Stockout rate distribution
so_vals = master[master['Stockout_Rate_Pct'] > 0]['Stockout_Rate_Pct']
axes[0].hist(so_vals, bins=30, color='#C00000', edgecolor='white')
axes[0].set_title(f'Stockout Rate Distribution\n({len(so_vals)} articles with ≥1 stockout)')
axes[0].set_xlabel('Stockout Rate (%)')
axes[0].set_ylabel('Articles')

# Fill rate by ABC × HML
fr_pivot = master.pivot_table(index='ABC', columns='HML', values='Fill_Rate_Pct', aggfunc='mean')
sns.heatmap(fr_pivot, annot=True, fmt='.1f', cmap='RdYlGn', vmin=80, vmax=100,
            linewidths=1, ax=axes[1], cbar_kws={'label': 'Fill Rate %'})
axes[1].set_title('Average Fill Rate % by ABC × HML')
axes[1].set_xlabel('HML'); axes[1].set_ylabel('ABC')

# Orders per year by HML
opy_data = master.groupby('HML')['Avg_Orders_Year'].mean().reindex(['H','M','L'])
colors_hml = ['#C00000','#FF9900','#70AD47']
axes[2].bar(opy_data.index, opy_data.values, color=colors_hml, edgecolor='white')
axes[2].set_title('Avg Orders per Year by HML')
axes[2].set_xlabel('HML Class'); axes[2].set_ylabel('Orders/Year')

plt.tight_layout()
plt.savefig('simulation_overview.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🚨 Step 7 — Prioritized action plan

In [ ]:
# ── 7.1 Priority scoring ─────────────────────────────────────────────────
# Score each article: higher = more urgent to fix
# Components: stockout rate + ABC weight + HML weight + CV

abc_weight = {'B': 3, 'C': 1}
hml_weight = {'H': 3, 'M': 2, 'L': 1}

master['Priority_Score'] = (
    master['Stockout_Rate_Pct'] * 0.5
    + master['ABC'].map(abc_weight).fillna(1) * 10
    + master['HML'].map(hml_weight).fillna(1) * 8
    + master['CV'].fillna(1).clip(upper=3) * 5
)

master = master.sort_values('Priority_Score', ascending=False)

# Action flag
def action_flag(row):
    if row['ABC'] == 'B' and row['HML'] == 'H' and row['Stockout_Rate_Pct'] > 0:
        return '🔴 URGENT — Review immediately'
    if row['ABC'] == 'B' and row['Stockout_Rate_Pct'] > 10:
        return '🟠 HIGH — Review this month'
    if row['HML'] == 'H' and row['Stockout_Rate_Pct'] > 20:
        return '🟠 HIGH — Frequent item at risk'
    if row['Stockout_Rate_Pct'] > 30:
        return '🟡 MEDIUM — Adjust ROP'
    if row['Stockout_Rate_Pct'] > 0:
        return '🟢 LOW — Monitor'
    return '✅ OK'

master['Action'] = master.apply(action_flag, axis=1)

print('=== TOP 20 PRIORITY ARTICLES ===')
display(master[['Article','Désignation article','ABC','HML','XYZ',
                'Stockout_Rate_Pct','Fill_Rate_Pct','SafetyStock',
                'ROP','EOQ','Priority_Score','Action']].head(20)
        .style.background_gradient(subset=['Priority_Score'], cmap='Reds')
        .bar(subset=['Stockout_Rate_Pct'], color='#FF4C4C')
        .format({'Priority_Score':'{:.1f}', 'Stockout_Rate_Pct':'{:.1f}%',
                 'Fill_Rate_Pct':'{:.1f}%','SafetyStock':'{:.1f}',
                 'ROP':'{:.1f}','EOQ':'{:.1f}'}))

In [ ]:
# ── 7.2 Action plan summary ───────────────────────────────────────────────
action_summary = master['Action'].value_counts().rename('Count').reset_index()
action_summary.columns = ['Action', 'Count']
print('=== ACTION PLAN SUMMARY ===')
display(action_summary)

print(f"\nTotal articles requiring attention: {(master['Stockout_Rate_Pct']>0).sum()}")
print(f"Total stock investment (optimized): TND {master['AvgStockValue_TND'].sum():,.0f}")
print(f"Investment in safety stock only   : TND {(master['SafetyStock']*master['UnitCost']).sum():,.0f}")

---
## 📤 Step 8 — Export results to Excel

In [ ]:
# ── 8.1 Write multi-sheet Excel report ───────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

OUTPUT_FILE = 'SOTACIB_Stock_Plan.xlsx'

def style_header(cell, bg='1F3864', fg='FFFFFF'):
    cell.font      = Font(bold=True, color=fg, size=10, name='Arial')
    cell.fill      = PatternFill('solid', fgColor=bg)
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    thin = Side(style='thin', color='BFBFBF')
    cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

def write_df_to_sheet(ws, df, header_bg='1F3864', freeze=True):
    """Write a DataFrame to a worksheet with styled headers."""
    # Headers
    for j, col in enumerate(df.columns):
        c = ws.cell(1, j+1, col)
        style_header(c, bg=header_bg)
        ws.column_dimensions[get_column_letter(j+1)].width = max(12, min(len(str(col))+4, 35))
    if freeze:
        ws.freeze_panes = 'A2'
    # Data rows
    alt = 'F2F2F2'
    thin = Side(style='thin', color='BFBFBF')
    for ri, row_data in enumerate(df.itertuples(index=False)):
        bg = alt if ri % 2 == 0 else 'FFFFFF'
        for ci, val in enumerate(row_data):
            c = ws.cell(ri+2, ci+1, val if not (isinstance(val, float) and np.isnan(val)) else '')
            c.font      = Font(size=9, name='Arial')
            c.fill      = PatternFill('solid', fgColor=bg)
            c.alignment = Alignment(vertical='center',
                                    horizontal='left' if ci in [1] else 'center')
            c.border    = Border(left=thin, right=thin, top=thin, bottom=thin)
            if isinstance(val, float) and not np.isnan(val):
                c.number_format = '#,##0.00'

wb = Workbook()

# ── Sheet 1: Full optimization table ─────────────────────────────────────
ws1 = wb.active
ws1.title = 'Optimization Parameters'
cols_opt = ['Article','Désignation article','Unit','ABC','HML','XYZ','Source',
            'UnitCost','LeadTime','ServiceLevel',
            'AvgMonthlyDemand','StdMonthlyDemand','CV',
            'SafetyStock','ROP','EOQ','StockMin','StockMax',
            'OrdersPerYear','AvgStockValue_TND']
write_df_to_sheet(ws1, master[cols_opt].reset_index(drop=True))

# ── Sheet 2: Simulation results ───────────────────────────────────────────
ws2 = wb.create_sheet('Simulation Results')
cols_sim = ['Article','Désignation article','ABC','HML','XYZ',
            'TotalConsumption','ActiveMonths',
            'Stockout_Months','Stockout_Rate_Pct','Fill_Rate_Pct',
            'Total_Orders','Avg_Orders_Year','Priority_Score','Action']
write_df_to_sheet(ws2, master[cols_sim].reset_index(drop=True), header_bg='2E75B6')

# Colour-code stockout rate
for ri in range(2, len(master)+2):
    rate = ws2.cell(ri, 9).value  # Stockout_Rate_Pct column
    if isinstance(rate, (int, float)):
        if rate > 30:
            ws2.cell(ri, 9).fill = PatternFill('solid', fgColor='FF4C4C')
            ws2.cell(ri, 9).font = Font(bold=True, color='FFFFFF', size=9)
        elif rate > 10:
            ws2.cell(ri, 9).fill = PatternFill('solid', fgColor='FFD966')

# ── Sheet 3: Priority action plan ─────────────────────────────────────────
ws3 = wb.create_sheet('Action Plan')
urgent = master[master['Action'].str.contains('URGENT|HIGH', na=False)]
cols_act = ['Article','Désignation article','ABC','HML','XYZ',
            'UnitCost','LeadTime','SafetyStock','ROP','EOQ',
            'Stockout_Rate_Pct','Fill_Rate_Pct','Priority_Score','Action']
write_df_to_sheet(ws3, urgent[cols_act].reset_index(drop=True), header_bg='C00000')

# ── Sheet 4: Summary by ABC × HML ─────────────────────────────────────────
ws4 = wb.create_sheet('Summary ABC-HML')
summary_full = (
    master.groupby(['ABC','HML'])
    .agg(
        Articles             = ('Article',           'count'),
        Avg_UnitCost         = ('UnitCost',          'mean'),
        Avg_LeadTime         = ('LeadTime',          'mean'),
        Avg_SafetyStock      = ('SafetyStock',       'mean'),
        Avg_ROP              = ('ROP',               'mean'),
        Avg_EOQ              = ('EOQ',               'mean'),
        Total_StockValue     = ('AvgStockValue_TND', 'sum'),
        Avg_Stockout_Rate    = ('Stockout_Rate_Pct', 'mean'),
        Avg_Fill_Rate        = ('Fill_Rate_Pct',     'mean'),
        Articles_w_Stockout  = ('Stockout_Months',   lambda x: (x>0).sum()),
    )
    .round(2)
    .reset_index()
)
write_df_to_sheet(ws4, summary_full, header_bg='404040')

wb.save(OUTPUT_FILE)
print(f'✅  Excel report saved → {OUTPUT_FILE}')
print(f'   Sheets: {[ws.title for ws in wb.worksheets]}')

In [ ]:
# ── 8.2 Download everything ───────────────────────────────────────────────
from google.colab import files
import os

# Download Excel report
files.download(OUTPUT_FILE)

# Download charts
for chart in ['demand_patterns.png','abc_hml_matrix.png',
              'planning_params.png','simulation_overview.png']:
    if os.path.exists(chart):
        files.download(chart)
        print(f'⬇  {chart}')

print('\n✅  All files downloaded')

---
## 🔍 Step 9 — Deep-dive: single article analysis (interactive)

In [ ]:
# ── Change ARTICLE_CODE to inspect any article ────────────────────────────
ARTICLE_CODE = '83520000'   # <-- edit this

row = master[master['Article'].astype(str) == str(ARTICLE_CODE)]
if row.empty:
    print(f'Article {ARTICLE_CODE} not found')
else:
    row = row.iloc[0]
    print('═'*60)
    print(f'  ARTICLE  : {row["Article"]}')
    print(f'  NAME     : {row["Désignation article"]}')
    print(f'  CLASS    : ABC={row["ABC"]}  HML={row["HML"]}  XYZ={row["XYZ"]}')
    print(f'  UNIT     : {row["Unit"]}   COST: TND {row["UnitCost"]:,.3f}')
    print(f'  SOURCE   : {row["Source"]}   Lead Time: {row["LeadTime"]:.0f} days')
    print('─'*60)
    print(f'  DEMAND   : avg {row["AvgMonthlyDemand"]:,.2f}/month  |  std {row["StdMonthlyDemand"]:,.2f}  |  CV {row["CV"]:,.2f}')
    print(f'  SERVICE  : {row["ServiceLevel"]}  (Z={row["Z_Factor"]})')
    print('─'*60)
    print(f'  SAFETY STOCK : {row["SafetyStock"]:,.2f} units')
    print(f'  ROP          : {row["ROP"]:,.2f} units   ← reorder when stock hits this')
    print(f'  EOQ          : {row["EOQ"]:,.2f} units   ← order this quantity each time')
    print(f'  STOCK MIN    : {row["StockMin"]:,.2f} units')
    print(f'  STOCK MAX    : {row["StockMax"]:,.2f} units')
    print(f'  ORDERS/YEAR  : {row["OrdersPerYear"]:,.1f}')
    print('─'*60)
    print(f'  SIMULATION   : {row["Stockout_Months"]} stockout months ({row["Stockout_Rate_Pct"]}%)')
    print(f'  FILL RATE    : {row["Fill_Rate_Pct"]}%')
    print(f'  ACTION       : {row["Action"]}')
    print('═'*60)

    # Monthly consumption chart
    if str(ARTICLE_CODE) in pivot.index:
        art_data = pivot.loc[str(ARTICLE_CODE)]
        fig, ax = plt.subplots(figsize=(14, 4))
        ax.bar(art_data.index, art_data.values, color='#2E75B6', alpha=0.8, width=20)
        ax.set_title(f'Monthly Consumption — {row["Désignation article"][:60]}')
        ax.set_xlabel('Month')
        ax.set_ylabel('Units consumed')
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))
        plt.tight_layout()
        plt.show()

---
## 📋 Step 10 — Sensitivity analysis: impact of service level on SS investment

In [ ]:
# ── How does changing service level affect total SS investment? ───────────
z_range  = np.arange(1.0, 2.6, 0.1)
sl_pct   = [f'{(2*(1 - 0.5*(1+np.math.erf((z)/np.sqrt(2))))*100):.1f}%'
             if hasattr(np, 'math') else f'Z={z:.1f}' for z in z_range]

results_sens = []
for z in z_range:
    total_ss_val = (
        z * master['StdDailyDemand'] * np.sqrt(master['LeadTime'].clip(lower=1))
        * master['UnitCost']
    ).sum()
    results_sens.append({'Z': z, 'Total_SS_Investment_TND': total_ss_val})

df_sens = pd.DataFrame(results_sens)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_sens['Z'], df_sens['Total_SS_Investment_TND'] / 1e6,
        color='#C00000', linewidth=2.5, marker='o', markersize=5)

# Mark current levels
for abc, sl in SERVICE_LEVELS.items():
    z_cur = sl['z']
    val_cur = df_sens[df_sens['Z'].round(2) == round(z_cur,2)]['Total_SS_Investment_TND']
    if not val_cur.empty:
        ax.axvline(z_cur, linestyle='--', alpha=0.6,
                   color='#2E75B6' if abc=='B' else '#FF9900',
                   label=f'Class {abc}: Z={z_cur} ({sl["label"]})')

ax.set_title('Sensitivity: Service Level vs Safety Stock Investment', fontsize=13, fontweight='bold')
ax.set_xlabel('Z-Factor (Service Level)')
ax.set_ylabel('Total SS Investment (TND Millions)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'TND {x:.1f}M'))
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('sensitivity_ss.png', dpi=150, bbox_inches='tight')
plt.show()
print('The curve shows how much more (or less) you invest in SS per Z-factor unit.')

---
## ✅ Summary of outputs

| File | Content |
|---|---|
| `SOTACIB_Stock_Plan.xlsx` | 4-sheet Excel: parameters, simulation, action plan, summary |
| `demand_patterns.png` | CV distribution, XYZ pie, active months |
| `abc_hml_matrix.png` | ABC×HML heatmap + strategy grid |
| `planning_params.png` | SS / ROP / EOQ by segment |
| `simulation_overview.png` | Stockout rates, fill rates, orders/year |
| `sensitivity_ss.png` | Service level vs SS investment curve |

### Key formulas used
```
Safety Stock  SS  = Z × σd × √LT
Reorder Point ROP = D̄_daily × LT + SS
EOQ               = √(2 × D_annual × Order_cost / Holding_cost)
Stock Min         = ROP
Stock Max         = ROP + EOQ
Fill Rate         = 1 − (Unfilled demand / Total demand)
```

> **Next steps:** Upload your current stock levels and I can add a column showing which articles are **already below ROP today** and need immediate reordering.